Python translation of Felippa's IFEM Chapter 5, originally written in Mathematica.

In [11]:
import math
import numpy as np

def ElemStiff2DTwoNodeBar(coords, E, A):
    """
    INPUT: 2 sets of 2D coords, youngs modulus and area
    PROCESS: calculates the element stiffness matrix, for use in a global master matrix assembly
    OUTPUT: Ke, the element stiffness matrix.
    """
    x1, y1 = coords[0]
    x2, y2 = coords[1]

    dx = x2 - x1
    dy = y2 - y1

    L = math.sqrt(dx**2 +dy**2)
    c = dx / L 
    s = dy / L

    Ke = ((E * A) / L) * np.array([
        [c**2, c*s, -c**2, -s*c],
        [c*s, s**2, -s*c, -s**2],
        [-c**2, -s*c, c**2, s*c],
        [-s*c, -s**2, s*c, s**2]
    ])
    
    return Ke

def MergeElemIntoMasterStiff(Ke, eftab, Kinp):
    """
    INPUT: Ke element stiff. matrix, 
           eftab (column of Element Freedom Table (EFT) appropiate to the member being merged; list with 4 int),
           Kinp (6 x 6 master stiffness matrix)
    PROCESS: creates a copy of the master matrix, then iterates over the element matrix and adds the Ke entry into correct mapping, based on EFT.
    OUTPUT: Master stiffness matrix updated with element entries.
    """
    K = Kinp.copy()
    ### Position 0, 1, 2, 3 in EFT list gives respectively x1, y1, x2, y2
    for i in range(len(Ke)):
        for j in range(len(Ke)):
            K[eftab[i]-1, eftab[j]-1] += Ke[i][j]
    return K

def AssembleMasterStiffOfExampleTruss():
    """
    INPUT: none
    PROCESS: computes the global element stiffness for each nodes and merges them into the free-free master stiffness matrix
    OUTPUT: free free master stiffness matrix.
    """
    K = np.zeros((6,6)) 
    Ke = ElemStiff2DTwoNodeBar([[0,0],[10,0]], 100, 1)
    K = MergeElemIntoMasterStiff(Ke, [1, 2, 3, 4], K)
    
    Ke = ElemStiff2DTwoNodeBar([[10,0],[10,10]], 100, 0.5)
    K = MergeElemIntoMasterStiff(Ke, [3, 4, 5, 6], K)
    
    Ke = ElemStiff2DTwoNodeBar([[0,0],[10,10]], 100, 2*math.sqrt(2))
    K = MergeElemIntoMasterStiff(Ke, [1, 2, 5, 6], K)

    return K


def ModifiedMasterStiffForDBC(pdof, K):
    """
    INPUT: pdof (list of DoF that are boundary cond, by global number), K
    PROCESS: modifies K to account boundary conditions done by clearing rows, cols and placing 1's in diagonal
    OUTPUT: Kmod
    """
    Kmod = K.copy()
    for i in pdof:
        i = i - 1  
        Kmod[i, :] = 0
        Kmod[:, i] = 0
        Kmod[i, i] = 1
    return Kmod

def ModifiedMasterForcesForDBC(pdof, f):
    """
    INPUT: pdof (list of DoF that are boundary cond, by global number), f (force vector)
    PROCESS: modifies force vector to account for boundary conditions.
    OUTPUT: Fmod
    """
    fmod = f.copy()
    for i in pdof:
        i = i- 1
        fmod[i] = 0
    return fmod

def IntForce2DTwoNodeBar(coords, E, A, eftab, u):
    """
    INPUT: coords, E, A, EFT for specific element, displacement u
    PROCESS: convers u into local coordinates (ubar), calculates the displacement (x2-x1), divides by length to obtain strain, gets axial internal force.
    OUTPUT: axial internal force
    """
    x1, y1 = coords[0]
    x2, y2 = coords[1]

    dx = x2 - x1
    dy = y2 - y1

    L = math.sqrt(dx**2 +dy**2)
    c = dx / L 
    s = dy / L

    # as Felippa says, calculating y displacements is redudant, "kept to illustrate the general backtransformation of global to local displacements"
    ix, iy, jx, jy = [eftab[0]-1, eftab[1]-1, eftab[2]-1, eftab[3]-1]

    ubar = [c*u[ix] + s*u[iy], 
            -s*u[ix] + c*u[iy],
            c*u[jx] + s*u[jy],
            -s*u[jx] + c*u[jy]]

    
    e = (ubar[2]- ubar[0]) / L
    return E * A * e

def IntForcesofExampleTruss(u):
    """
    INPUT: displacement vector u
    PROCESS: calls IntForce2DTwoNodeBar for each element in the example 3-bar truss, obtains its axial internal force, adds them to a force vector
    OUTPUT: internal forces vector f
    """
    f = np.zeros(3)
    f[0] = IntForce2DTwoNodeBar([[0,0], [10,0]], 100, 1, [1,2,3,4], u)
    f[1] = IntForce2DTwoNodeBar([[10,0], [10,10]], 100, 0.5, [3,4,5,6], u)
    f[2] = IntForce2DTwoNodeBar([[0,0], [10,10]], 100, 2*math.sqrt(2), [1,2,5,6], u)
    return f

# Driving program. 
# Steps: 1. assembles MasterStiffOfExampleTruss 2. DBC by ModifiedMasterStiffForDBC and ModifiedMasterForcesForDBC 3. Solves system, recovers forces.


f = np.array([0, 0, 0, 0, 2, 1])
K = AssembleMasterStiffOfExampleTruss()
Kmod = ModifiedMasterStiffForDBC([1,2,4], K)
fmod = ModifiedMasterForcesForDBC([1,2,4], f)
u = np.linalg.solve(Kmod, fmod)
print(f"Computed nodal displacements:{u}")
f_recovered = K @ u
print(f"External node forces including reactions: {f_recovered}")
p = IntForcesofExampleTruss(u)
print(f"Internal member forces: {p}")


''' As a crosscheck, here are the values obtained by Felippa (see pg 76):
Computed nodal displacements:{0, 0, 0, 0, 2/5, -1/5}
External node forces including reactions: {-2, -2, 0, 1, 2, 1}
Internal member forces: {0, -1, 2*sqrt(2)}
''';

Computed nodal displacements:[ 0.   0.   0.   0.   0.4 -0.2]
External node forces including reactions: [-2. -2.  0.  1.  2.  1.]
Internal member forces: [ 0.         -1.          2.82842712]
